# N035 · 区间合并、插入与交集

**目标：** 明确开闭语义并按排序后的端点处理区间。

**先修：** N025, N017。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 重叠判定；排序扫描；区间交集；相邻端点；规范化输出。

## 从问题到算法

区间合并先按左端排序。当前段的右端是已合并区间的最远右端；新区间开始于它之后才需要另开一段。闭区间同端点相接也算相交。两个各自有序互不重叠的列表求交时，较早结束的区间以后不可能再与当前另一区间产生新交集，应移动它。

## 最小实现

**本章接口：** `merge_intervals(intervals)`、`insert_interval(intervals, new)`、`interval_intersection(a, b)`

In [1]:
def merge_intervals(intervals):
    merged=[]
    for left,right in sorted(intervals):
        if left>right: raise ValueError('reversed interval')
        if not merged or left>merged[-1][1]: merged.append([left,right])
        else: merged[-1][1]=max(merged[-1][1],right)
    return merged

def insert_interval(intervals,new):
    return merge_intervals([*intervals,list(new)])

def interval_intersection(a,b):
    i=j=0; answer=[]
    while i<len(a) and j<len(b):
        left=max(a[i][0],b[j][0]); right=min(a[i][1],b[j][1])
        if left<=right: answer.append([left,right])
        if a[i][1]<b[j][1]: i+=1
        else: j+=1
    return answer

## 正确性、前提与复杂度

排序后未来区间的左端不会回退，已经封闭的段不可能被后续桥接。求交时较早结束者与之后所有另一侧区间也不会再相交，移动安全。

**代价：** 合并与本章简洁插入 O(n log n)，已排序输入可把插入进一步做成 O(n)；交集 O(n+m)，输出空间与答案大小一致。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[[1,3],[2,6],[8,10],[10,12]]
show_table(['处理前缀','合并后'],[(a[:i],merge_intervals(a[:i])) for i in range(1,len(a)+1)])

处理前缀,合并后
"[[1, 3]]","[[1, 3]]"
"[[1, 3], [2, 6]]","[[1, 6]]"
"[[1, 3], [2, 6], [8, 10]]","[[1, 6], [8, 10]]"
"[[1, 3], [2, 6], [8, 10], [10, 12]]","[[1, 6], [8, 12]]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert merge_intervals([[1,3],[2,6],[8,10]])==[[1,6],[8,10]]
assert merge_intervals([[1,4],[2,3],[4,5]])==[[1,5]]
assert insert_interval([[1,2],[5,7]],[2,6])==[[1,7]]
assert interval_intersection([[1,2]],[[2,3]])==[[2,2]]
from random import Random
rng=Random(35)
for _ in range(100):
    a=[sorted([rng.randrange(6),rng.randrange(6)]) for _ in range(5)]
    m=merge_intervals(a)
    assert all(m[i][1]<m[i+1][0] for i in range(len(m)-1))
    for x in [i/2 for i in range(11)]:
        assert any(l<=x<=r for l,r in a)==any(l<=x<=r for l,r in m)
print('N035: 所有本章断言通过')

N035: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 比较闭区间与半开区间的相邻合并条件。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 输出覆盖总长度。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 56. Merge Intervals（canonical）
- 57. Insert Interval（canonical）
- 986. Interval List Intersections（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。